In [ ]:
import os
os.environ["HF_HOME"] = "./hf_home"
os.makedirs("./hf_home", exist_ok=True)

In [ ]:
!pip install git+https://github.com/huggingface/transformers.git
!pip install accelerate soundfile
!pip install torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu126

In [ ]:
from huggingface_hub import login, snapshot_download

login("hf_...")  # トークン入れてください

snapshot_download(
    repo_id="ibm-granite/granite-speech-4.1-2b",
    local_dir="./hf_home/granite-speech",
    ignore_patterns=["*.msgpack", "*.h5"]
)

print("ダウンロード完了！")

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForSpeechSeq2Seq

model_path = "./hf_home/granite-speech"

processor = AutoProcessor.from_pretrained(model_path)
print("Processor OK")

model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_path,
    dtype=torch.float16,
    device_map="auto"
)
model.eval()
print("Model OK")

In [ ]:
# ここに Jupyter Notebookエディタ の「コードスニペット」機能から
# 音声ファイル（レコーディング1.wav）の COS 取得コードを挿入してください
# 挿入後、以下の保存コードと同じセルで実行してください

with open("./レコーディング1.wav", "wb") as f:
    f.write(streaming_body_1.read())
print("レコーディング1.wav 保存完了")

In [ ]:
import torchaudio
import time

tokenizer = processor.tokenizer
user_prompt = "<|audio|>transcribe the speech with proper punctuation and capitalization."
chat = [{"role": "user", "content": user_prompt}]
prompt = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)

results = {}
total_start = time.time()

for i in range(1, 2):  # 1だけ
    filename = f"./レコーディング{i}.wav"
    wav, sr = torchaudio.load(filename, normalize=True)
    print(f"レコーディング{i}: shape={wav.shape}, sr={sr}")
    
    start = time.time()
    model_inputs = processor(prompt, wav, device="cuda", return_tensors="pt").to("cuda")
    model_outputs = model.generate(
        **model_inputs,
        max_new_tokens=512,
        repetition_penalty=1.3,
        no_repeat_ngram_size=5,
    )
    elapsed = time.time() - start
    
    output_ids = model_outputs[0][model_inputs["input_ids"].shape[1]:]
    result = tokenizer.decode(output_ids, skip_special_tokens=True)
    results[i] = {"text": result, "time": elapsed}
    print(f"  → {elapsed:.1f}秒: {result}")

print(f"\n総処理時間: {time.time()-total_start:.1f}秒")

In [ ]:
for i, data in results.items():
    with open(f"./レコーディング{i}_granite_transcript.txt", "w", encoding="utf-8") as f:
        f.write(data["text"])
    print(f"レコーディング{i}: 保存完了")

In [ ]:
import base64
from IPython.display import HTML, display

for i in range(1, 2):  # 1だけ
    filename = f"./レコーディング{i}_granite_transcript.txt"
    with open(filename, "r", encoding="utf-8") as f:
        content = f.read()
    b64 = base64.b64encode(content.encode()).decode()
    html = f'<a href="data:text/plain;base64,{b64}" download="レコーディング{i}_granite_transcript.txt">📄 レコーディング{i}_granite_transcript.txt</a>'
    display(HTML(html))